# Module 1: The Inference Stack and Runtime Landscape

In Module 0 you connected to your own vLLM endpoint and sent one request. This module makes the case for that arrangement. You own the layer your agents run on: a model you serve yourself with [vLLM](https://docs.vllm.ai) on a dedicated [Akamai Cloud GPU](https://www.linode.com/products/gpu/), instead of renting tokens from a hosted API.

## Learning objectives
- Resolve your connection settings from the environment and reach your own vLLM endpoint
- Trace one streamed request and read its time to first token and tokens per second
- See why the engine is a real layer: the same model is fast or slow depending on the runtime and hardware under it
- Point a minimal agent at your endpoint and confirm its model is the GPU you own
- See that an agent is many requests, where the slow step sets the pace
- Compare a hosted API against your self-hosted server on cost, data path, and control
- Place vLLM among the other runtimes and know why this workshop runs it

## Prerequisites
- Finished Module 0, or otherwise confirmed your vLLM endpoint and namespace work
- Your vLLM Deployment is named `vllm` and serves `Qwen/Qwen3-4B`
- About 15 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/) &middot; [OpenAI chat API](https://platform.openai.com/docs/api-reference/chat) &middot; [SGLang](https://github.com/sgl-project/sglang) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/)

## Inference stack design basics

A request to a self-hosted model passes through a few layers. Knowing them tells you where to look when something is slow or expensive.

- The runtime is the server that loads the model and answers requests. This workshop runs vLLM. Others are SGLang, TensorRT-LLM, and llama.cpp.
- The API surface is how you talk to it. vLLM speaks the OpenAI API, so your client code points at it with one change, the base URL.
- The scheduler decides which requests run now and which wait, and it batches requests together to keep the GPU busy.
- The KV cache holds the running attention state for every in-flight request. It lives in the same GPU memory as the model weights. Module 2 sizes it, Module 3 builds it by hand.
- The metrics endpoint reports what the server is doing right now: queue depth, cache use, and latency. You read it in every later module.

![Inside your namespace: your JupyterLab pod calls a vLLM Service on a dedicated GPU node and reads its metrics endpoint](images/01_inference_stack_architecture.png)

The notebook runs on a CPU and never loads model weights. Everything that needs a GPU happens in the vLLM pod, which you reach over HTTP.

## 1. Setup

Install the small client dependencies and make the repo's `common/` package importable. The notebook works whether Jupyter starts in this module folder or the repo root. `print_settings()` shows the resolved connection values and never prints the API key.

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31"

In [ ]:
# Imports, settings, and paths used throughout the module.
import os, sys, time
from pathlib import Path

if Path("../common").exists():
    REPO_ROOT = Path("..")
else:
    REPO_ROOT = Path(".")
sys.path.insert(0, str(REPO_ROOT.resolve()))

from common.config import get_settings, print_settings, build_client
from common import foundation

settings = get_settings()
settings.model_name = foundation.served_model_name(settings)  # use the model the server actually serves
print_settings(settings)

**What you should see:** your `VLLM_HOST`, the derived metrics URL, `MODEL_NAME` (`Qwen/Qwen3-4B`), and `NAMESPACE`. On the hosted workshop these are filled in for you. If `VLLM_HOST` still shows the in-cluster default and you are off cluster, set the variables and restart the kernel.

## 2. Reach the server you own

Build an OpenAI client pointed at your endpoint and send one chat completion. `build_client` is the standard `openai` client with two conveniences for this server: it turns off Qwen3 thinking for clean measurements, and it drops an empty `tools=[]` that vLLM rejects. The model id comes from your environment, so nothing here is hardcoded to one deployment.

In [ ]:
# Requires a live vLLM endpoint. Send one chat completion through the server you control.
client = build_client(settings)
resp = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": "In one sentence, what is an inference server?"}],
    max_tokens=64,
    temperature=0.0,
)
print(resp.choices[0].message.content)
print("\nserved by:", settings.model_name)

**What you should see:** one sentence generated by a server you control. A connection error means `VLLM_HOST` is wrong or the pod is not ready. A 404 on the model means `MODEL_NAME` does not match what the server loaded; check Module 0.

## 3. Trace one request, end to end

Follow a single request through the server. The client sends a prompt. The scheduler admits it. Prefill reads the whole prompt in one pass and fills the KV cache. Decode then generates one token at a time and streams each back. Two numbers name the two phases: time to first token is prefill, time per output token is decode. Stream one request and measure both. Module 3 measures these properly against the server's own histograms; here you see the shape from the client side.

In [ ]:
# Requires a live vLLM endpoint. Stream one request and time the first token (prefill) and the gaps (decode).
start = time.time()
first = None
stamps = []
stream = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": "List the numbers 1 through 50, one per line."}],
    max_tokens=120,
    temperature=0.0,
    stream=True,
)
for chunk in stream:
    if not chunk.choices or not chunk.choices[0].delta.content:
        continue
    now = time.time()
    if first is None:
        first = now
    stamps.append(now)

ttft = (first - start) if first else 0.0
gaps = [b - a for a, b in zip(stamps, stamps[1:])]
tpot = (sum(gaps) / len(gaps)) if gaps else 0.0
print(f"time to first token (prefill) : {ttft * 1000:.0f} ms")
if tpot:
    print(f"time per output token (decode): {tpot * 1000:.1f} ms  ({1 / tpot:.0f} tokens/s)")

**What you should see:** a TTFT in the low hundreds of milliseconds and a decode rate of tens of tokens per second on the 4B model. The decode rate has a hard ceiling set by how fast the GPU reads the weights once per token, which you compute in Module 2. If it is well below that, the shared card is busy or your prompt is unusually long; re-run and check. Module 3 explains why prefill is fast per token and decode is the slow part.

## 4. The engine is a real layer

The model is not the whole story. The same weights on a different engine and different card run at a very different speed. To make that concrete, compare two paths on the same kind of prompt: a small model on a CPU with a plain generate loop, and your 4B on a GPU behind vLLM. This is not a fair benchmark; one side has no serving engine. The point is that the engine and the card under the same API change the result by orders of magnitude.

In [ ]:
# The CPU path is a representative figure, not a live run: loading even a 0.6B model into this
# 1 GB notebook pod would kill the kernel. A 0.6B in full precision on one CPU core does only
# single-digit tokens per second, with no GPU and no serving engine.
print("representative CPU baseline: a 0.6B on one CPU core does single-digit tok/s")

**What you should see:** the representative CPU baseline, single-digit tokens per second. No GPU, no serving engine, a small model, and it crawls. The notebook loads no model; the figure is stated, not measured.

In [ ]:
# Requires a live vLLM endpoint. The other path: your 4B on a GPU behind vLLM, same prompt, measured end to end.
prompt = "Explain what an LLM inference server does, in two sentences."
t0 = time.time()
resp = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=60,
    temperature=0.0,
)
dt = time.time() - t0
n = resp.usage.completion_tokens
print(f"{settings.model_name} on GPU (vLLM): {n} tokens in {dt:.2f}s = {n / dt:.1f} tok/s")
print("\nsame OpenAI API, different engine and card: single-digit CPU tok/s against tens here, orders of magnitude apart")

**What you should see:** the vLLM path finishing many times faster than that stated CPU baseline, even though it runs a much larger model. The CPU baseline is full precision with no optimized kernels, so the gap overstates the engine alone. A real engine comparison holds the model and hardware fixed and drives load, which is Module 7.

> NOTE: The 4B is the larger model here and still wins by a wide margin. Size did not make it fast. The GPU and the serving engine did.

## 5. Your first agent

The workshop is named for the agent, so meet it now. An agent is a model that can be given a job and tools. Here it is the `openai` client wrapped with a system prompt. This is the same Akamai Cloud Solutions Architect persona you deploy as a real Kubernetes Service in Module 9. Thinking is off here for a clean, fast answer; the deployed agent in Module 9 turns it on so it can reason and call a tool. Point it at your vLLM and ask what powers it.

In [ ]:
# Requires a live vLLM endpoint. A minimal agent: a system prompt plus a call to the model you own. No framework.
SYSTEM_PROMPT = (
    "You are the Akamai Cloud Solutions Architect agent. You help developers with "
    "Akamai Cloud and the Kubernetes cluster you run in. You are tactical and concise, "
    "developer to developer. In scope: Compute, LKE, Object Storage, Cloud networking, "
    "GPUs, and AI inference. You run on self-hosted inference served by vLLM. If asked "
    "what powers you, say so."
)

def agent(message):
    resp = client.chat.completions.create(
        model=settings.model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": message},
        ],
        max_tokens=200,
        temperature=0.2,
    )
    return resp.choices[0].message.content.strip()

print(agent("What model and inference server are you running on?"))

**What you should see:** an answer that names self-hosted vLLM. The agent's brain is the GPU in your namespace, not a rented API. You spend the rest of the workshop making that brain faster and cheaper, then deploy this agent on top of it in Module 9.

## 6. An agent is many requests on your endpoint

The agent above answered in one shot. A real agent loops: it calls the model, reads a tool result, and calls again, re-sending a growing prompt each step. Run a few steps of that same solutions-architect agent, this time pricing GPUs in a ReAct loop, and watch each step arrive as its own request with its own time to first token while the prompt grows each turn.

![A chatbot is one request; an agent is a loop that re-prefills a growing prompt every step](images/01_agent_loop.png)

In [ ]:
# Requires a live vLLM endpoint. Run 4 steps of the solutions-architect agent, timing each.
records = foundation.sa_agent_loop(steps=4, obs_tokens=200, max_tokens=120)

print(f"{'step':>4} {'ttft_ms':>8} {'prompt_tok':>11} {'out_tok':>8}")
for r in records:
    print(f"{r['step']:>4} {str(r.get('ttft_ms')):>8} {str(r.get('prompt_tokens')):>11} {str(r.get('completion_tokens')):>8}")
print(f"\nthe agent made {len(records)} model calls; its wall clock is the sum of their steps")

**What you should see:** four steps, each its own request with its own TTFT, and `prompt_tokens` climbing each step. That climb is the context tax: the prompt grows every turn and never shrinks. The agent's total time is the sum of its steps, so the slowest step sets the pace for the whole chain. That is why you tune the worst step, not the average, and the later modules cut it with prefix caching and a KV budget that avoids preemption.

## 7. Renting versus owning, with your own numbers

Inference is the same operation either way: a prompt goes in, tokens come out. What differs is the path and the price. Read the `usage` object on a response, the token counts a provider would bill, then put your real volume into the cost cell and find where owning passes renting.

In [ ]:
# Requires a live vLLM endpoint. Read the token usage a provider would bill for one request.
resp = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": "Explain what an LLM inference server does, in two sentences."}],
    max_tokens=128,
    temperature=0.0,
)
print(resp.choices[0].message.content)
print("\nusage:", resp.usage)

In [ ]:
# Estimate the monthly bill both ways at YOUR volume. Edit these to match your workload.
requests_per_day = 200_000
input_tokens_per_request = 500
output_tokens_per_request = 300

# Hosted prices in dollars per million tokens. Replace with your provider's.
hosted_input_price = 0.15
hosted_output_price = 0.60

# Owned: one dedicated GPU instance, billed by the hour no matter the token count.
gpu_hourly = 1.50   # your Akamai Cloud GPU plan price per hour

in_tokens = requests_per_day * input_tokens_per_request * 30
out_tokens = requests_per_day * output_tokens_per_request * 30
hosted_monthly = (in_tokens / 1e6) * hosted_input_price + (out_tokens / 1e6) * hosted_output_price
owned_monthly = gpu_hourly * 24 * 30

print(f"hosted bill (rented): ${hosted_monthly:,.0f} / month, and it grows with every token")
print(f"owned GPU (fixed)   : ${owned_monthly:,.0f} / month, flat no matter the volume")
verdict = "owning is cheaper at this volume" if owned_monthly < hosted_monthly else "renting is cheaper at this volume"
print(f"=> {verdict}")

**What you should see:** a hosted bill that scales with every token, and a flat owned cost. There is a crossover, and you just found it for your volume.

Cost is the obvious axis. Three others matter as much in production.

- **Data residency.** Your self-hosted request never left the cluster. For regulated data or anything under a contractual boundary, that is the line between compliant and not.
- **Rate limits.** A hosted provider sets your throughput and can throttle you during a launch. On your own server the only ceiling is the card you provisioned, and you watch it coming in the metrics.
- **Control.** You pick the model, the precision, the context length, and the batching policy. You tune for your traffic instead of accepting defaults. That is the rest of this workshop.

## 8. The runtime landscape

vLLM is one of several runtimes. Knowing where it sits tells you what you are choosing and what you could switch to.

- **vLLM**: the broad default for GPU serving. PagedAttention KV cache, continuous batching, wide model and quantization support, OpenAI-compatible server. Best when you want to run many models fast with no per-model compile step.
- **SGLang**: best when requests share a long prefix, like multi-turn chat, agents, and RAG. Its prefix cache reuses the KV cache across requests, and it is fast at structured output.
- **TensorRT-LLM**: best at peak throughput and latency on NVIDIA hardware. NVIDIA only, with an upfront tuning cost, so it fits stable models in long-running production.
- **llama.cpp**: best for local, edge, and CPU-first inference. Runs heavily quantized models from a single file on a laptop or an Arm server.

All four expose an OpenAI-compatible server, so your client code moves between them. This workshop runs vLLM because it is the common denominator, and it carries the batching and KV cache ideas you measure all day. Ask your own server which models it serves.

In [ ]:
# Requires a live vLLM endpoint. Ask the server which models it serves.
import requests
root = settings.vllm_host.rstrip("/").removesuffix("/v1")
data = requests.get(
    f"{root}/v1/models",
    headers={"Authorization": f"Bearer {settings.api_key}"},
    timeout=10,
).json()
ids = [m["id"] for m in data.get("data", [])]
print("served models      :", ids)
print("matches MODEL_NAME :", settings.model_name in ids)

**What you should see:** your model id, and a match against `MODEL_NAME`. This is the same `/v1` surface SGLang, TensorRT-LLM, and llama.cpp expose, which is why switching runtimes does not rewrite your client.

## Things to know

- **The OpenAI-compatible surface is why this works.** Your agent code does not change when the server becomes yours. Only the base URL does.
- **The notebook pod has no GPU.** Inference runs in the vLLM pod on a GPU node. You drive and measure it over the network, which is how you operate it in production.
- **The engine is a real layer.** The same model on a different engine or card has a different speed. Owning the layer means you choose that engine and tune it.
- **The metrics URL is derived, not separate.** It is `VLLM_HOST` with `/metrics` in place of `/v1`. Module 3 starts reading it for real.

## Try it yourself

**Find your crossover.** Edit `requests_per_day` and the prices in section 7 until the hosted bill passes the cost of a dedicated GPU. That volume is where owning starts to pay. **Stretch:** add a second, larger model's prices and see how much sooner the line crosses.

**Give the agent a real question.** Ask the section 5 agent something in scope (an LKE or GPU question) and something out of scope, and see how the system prompt holds the line.

**Watch the prompt change TTFT.** Send a very short prompt and a very long one in section 3 and compare the time to first token. Prefill grows with prompt length.

## Summary

- You reached a vLLM endpoint you control and traced one request, naming time to first token (prefill) and time per output token (decode).
- You saw the engine is a real layer: a small model on a CPU crawled while your larger model on a GPU behind vLLM finished in a fraction of the time.
- You pointed a minimal agent at your endpoint and saw that an agent is many requests, where the slow step sets the pace.
- You compared renting against owning on cost, data path, and control, and found the crossover for your volume.
- You placed vLLM among the runtimes and confirmed the served model.

## Next

**Module 2: Units and the Memory Budget.** You can reach the server and you have seen where a request spends its time. Next you size what a model costs in memory: the weights, the KV cache, and the budget that decides how many users fit on one card, all from arithmetic.